In [1]:
import sys
import os

# Add project root directory to Python path
sys.path.append(os.path.abspath(".."))

In [2]:
from src.data_loader import load_data
from src.preprocessing import preprocess_reviews
from src.sentiment import (
    load_sentiment_model,
    analyze_sentiment
)

c:\Users\derese\fintech-review-analytics\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sentiment_pipeline = load_sentiment_model()

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1113.07it/s]


In [4]:
df = load_data(
    "../data/raw/bank_reviews_cleaned.csv"
)
df.columns

Index(['review_id', 'review', 'rating', 'date', 'bank', 'source'], dtype='object')

In [5]:
df = preprocess_reviews(df)

print(df.shape)

(1500, 6)


In [6]:
sample_review = df["review"].iloc[0]

analyze_sentiment(
    sample_review,
    sentiment_pipeline
)

0    positive
1    0.999592
dtype: object

In [7]:
sample_df = df.copy()

sample_df[
    ["sentiment_label", "sentiment_score"]
] = sample_df["review"].apply(
    lambda x: analyze_sentiment(
        x,
        sentiment_pipeline
    )
)

In [8]:
sample_df[
    [
        "review",
        "sentiment_label",
        "sentiment_score"
    ]
].head(10)

,review,sentiment_label,sentiment_score
0,wow,positive,0.999592
1,Good application,positive,0.999855
2,"Nice, but I can't get some recently transactio...",negative,0.987926
3,Very Secure but very poor interface and limite...,negative,0.999777
4,very nice 100%,positive,0.999864
5,good,positive,0.999816
6,Good to use,positive,0.999846
7,cbe,positive,0.996601
8,Cbe,positive,0.996601
9,best and secured,positive,0.999819


In [9]:
sample_df.to_csv(
    "../data/raw/sentiment_analysis_results.csv",
    index=False
)

print("Sentiment analysis completed.")

Sentiment analysis completed.


In [10]:
sample_df[
    [
        "review",
        "rating",
        "bank",
        "sentiment_label",
        "sentiment_score"
    ]
].head()

,review,rating,bank,sentiment_label,sentiment_score
0,wow,5,Commercial Bank of Ethiopia,positive,0.999592
1,Good application,2,Commercial Bank of Ethiopia,positive,0.999855
2,"Nice, but I can't get some recently transactio...",5,Commercial Bank of Ethiopia,negative,0.987926
3,Very Secure but very poor interface and limite...,1,Commercial Bank of Ethiopia,negative,0.999777
4,very nice 100%,5,Commercial Bank of Ethiopia,positive,0.999864


In [11]:
from src.vader_sentiment import (
    analyze_vader_sentiment
)

In [12]:
vader_df = df.copy()

vader_df[
    [
        "vader_sentiment_label",
        "vader_sentiment_score"
    ]
] = vader_df["review"].apply(
    analyze_vader_sentiment
)

In [13]:
print(vader_df.columns)

Index(['review_id', 'review', 'rating', 'date', 'bank', 'source',
       'vader_sentiment_label', 'vader_sentiment_score'],
      dtype='object')


In [14]:
vader_df = df.head(100).copy()

In [15]:
vader_df[
    ["sentiment_label", "sentiment_score"]
] = vader_df["review"].apply(
    lambda x: analyze_sentiment(
        x,
        sentiment_pipeline
    )
)

In [16]:
vader_df[
    [
        "vader_sentiment_label",
        "vader_sentiment_score"
    ]
] = vader_df["review"].apply(
    analyze_vader_sentiment
)

In [17]:
comparison_df = vader_df[
    [
        "review",
        "sentiment_label",
        "sentiment_score",
        "vader_sentiment_label",
        "vader_sentiment_score"
    ]
]

comparison_df.head(10)

,review,sentiment_label,sentiment_score,vader_sentiment_label,vader_sentiment_score
0,wow,positive,0.999592,positive,0.5859
1,Good application,positive,0.999855,positive,0.4404
2,"Nice, but I can't get some recently transactio...",negative,0.987926,positive,0.5927
3,Very Secure but very poor interface and limite...,negative,0.999777,negative,-0.9371
4,very nice 100%,positive,0.999864,positive,0.4754
5,good,positive,0.999816,positive,0.4404
6,Good to use,positive,0.999846,positive,0.4404
7,cbe,positive,0.996601,neutral,0.0000
8,Cbe,positive,0.996601,neutral,0.0000
9,best and secured,positive,0.999819,positive,0.7845


## Sentiment Score Aggregation

To better understand customer satisfaction patterns, sentiment scores were aggregated by bank and by star rating. 

The aggregation helps identify:
- Which banks receive more positive or negative customer feedback.
- The relationship between user ratings and predicted sentiment scores.
- Satisfaction trends across different review categories.

In [19]:
bank_sentiment = (
    sample_df
    .groupby("bank")["sentiment_score"]
    .mean()
    .reset_index()
)

bank_sentiment.columns = [
    "bank",
    "average_sentiment_score"
]

bank_sentiment

,bank,average_sentiment_score
0,Bank of Abyssinia,0.966775
1,Commercial Bank of Ethiopia,0.976783
2,Dashen Bank,0.975582


In [20]:
rating_sentiment = (
    sample_df
    .groupby("rating")["sentiment_score"]
    .mean()
    .reset_index()
)

rating_sentiment.columns = [
    "rating",
    "average_sentiment_score"
]

rating_sentiment

,rating,average_sentiment_score
0,1,0.979106
1,2,0.965342
2,3,0.977547
3,4,0.962809
4,5,0.972193


In [21]:
bank_rating_sentiment = (
    sample_df
    .groupby(
        ["bank", "rating"]
    )["sentiment_score"]
    .mean()
    .reset_index()
)

bank_rating_sentiment

,bank,rating,sentiment_score
0,Bank of Abyssinia,1,0.982077
1,Bank of Abyssinia,2,0.942200
2,Bank of Abyssinia,3,0.956045
3,Bank of Abyssinia,4,0.956412
4,Bank of Abyssinia,5,0.962167
5,Commercial Bank of Ethiopia,1,0.976288
6,Commercial Bank of Ethiopia,2,0.963019
7,Commercial Bank of Ethiopia,3,0.985833
8,Commercial Bank of Ethiopia,4,0.963283
9,Commercial Bank of Ethiopia,5,0.978075


## TF-IDF Keyword and N-Gram Extraction

TF-IDF (Term Frequency-Inverse Document Frequency) was used to identify important keywords and recurring phrases in customer reviews.

Both single keywords and bi-grams (two-word phrases) were extracted to capture meaningful business-related issues such as:
- "login error"
- "slow transfer"
- "good interface"

The extracted keywords were later grouped into broader business themes.

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

In [23]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50,
    ngram_range=(1, 2)
)

In [24]:
tfidf_matrix = tfidf.fit_transform(
    sample_df["review"]
)

In [25]:
keywords = tfidf.get_feature_names_out()

keywords

array(['access', 'account', 'amazing', 'app', 'application', 'apps',
       'bank', 'banking', 'banking app', 'best', 'best app', 'boa', 'cbe',
       'dashen', 'dashen bank', 'doesn', 'easy', 'easy use', 'ethiopia',
       'excellent', 'fast', 'fix', 'good', 'good app', 'great', 'like',
       'make', 'mobile', 'mobile banking', 'money', 'need', 'new', 'nice',
       'nice app', 'ok', 'open', 'problem', 'service', 'slow', 'super',
       'super app', 'time', 'transaction', 'transfer', 'update', 'use',
       'work', 'working', 'worst', 'ነው'], dtype=object)

In [26]:
keyword_df = pd.DataFrame(
    keywords,
    columns=["keyword"]
)

keyword_df.head(20)



,keyword
0,access
1,account
2,amazing
3,app
4,application
5,apps
6,bank
7,banking
8,banking app
9,best


## Theme Grouping Logic

The extracted keywords and n-grams were manually grouped into broader business themes based on semantic similarity and business relevance.

Examples:
- Keywords such as "login error", "password issue", and "verification failed" were grouped under **Account Access Issues**.
- Keywords such as "slow transfer", "transaction failed", and "payment delay" were grouped under **Transaction Performance**.
- Keywords related to interface usability and navigation were grouped under **UI & Design**.

This approach helps transform low-level textual patterns into actionable business insights.

In [27]:
from src.themes import identify_theme

In [28]:
keyword_df["theme"] = keyword_df[
    "keyword"
].apply(identify_theme)

keyword_df.head(20)

,keyword,theme
0,access,Other
1,account,Other
2,amazing,Other
3,app,Other
4,application,Other
5,apps,Other
6,bank,Other
7,banking,Other
8,banking app,Other
9,best,Other


In [29]:
theme_summary = (
    keyword_df["theme"]
    .value_counts()
    .reset_index()
)

theme_summary.columns = [
    "theme",
    "keyword_count"
]

theme_summary

,theme,keyword_count
0,Other,45
1,Transaction Performance,3
2,Customer Support,1
3,Feature Requests,1


In [30]:
cbe_reviews = sample_df[
    sample_df["bank"] ==
    "Commercial Bank of Ethiopia"
]

In [31]:
cbe_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20,
    ngram_range=(1, 2)
)

cbe_matrix = cbe_tfidf.fit_transform(
    cbe_reviews["review"]
)

cbe_keywords = cbe_tfidf.get_feature_names_out()

cbe_keywords

array(['app', 'application', 'bank', 'best', 'cbe', 'easy', 'excellent',
       'fast', 'good', 'like', 'mobile', 'nice', 'nice app', 'ok',
       'service', 'transfer', 'update', 'use', 'work', 'working'],
      dtype=object)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

In [32]:
def extract_bank_themes(df, bank_name):

    # Filter bank reviews
    bank_reviews = df[
        df["bank"] == bank_name
    ]

    # TF-IDF
    tfidf = TfidfVectorizer(
        stop_words="english",
        max_features=50,
        ngram_range=(1, 2)
    )

    tfidf_matrix = tfidf.fit_transform(
        bank_reviews["review"]
    )

    keywords = tfidf.get_feature_names_out()

    # Create dataframe
    keyword_df = pd.DataFrame(
        keywords,
        columns=["keyword"]
    )

    # Assign themes
    keyword_df["theme"] = keyword_df[
        "keyword"
    ].apply(identify_theme)

    # Count themes
    theme_summary = (
        keyword_df["theme"]
        .value_counts()
        .reset_index()
    )

    theme_summary.columns = [
        "theme",
        "keyword_count"
    ]

    return keyword_df, theme_summary

In [33]:
cbe_keywords, cbe_themes = extract_bank_themes(
    sample_df,
    "Commercial Bank of Ethiopia"
)

cbe_themes

,theme,keyword_count
0,Other,45
1,Transaction Performance,3
2,Customer Support,1
3,Feature Requests,1


In [34]:
boa_keywords, boa_themes = extract_bank_themes(
    sample_df,
    "Bank of Abyssinia"
)

boa_themes

,theme,keyword_count
0,Other,46
1,Transaction Performance,2
2,Customer Support,1
3,Feature Requests,1


In [35]:
dashen_keywords, dashen_themes = extract_bank_themes(
    sample_df,
    "Dashen Bank"
)

dashen_themes

,theme,keyword_count
0,Other,45
1,Transaction Performance,2
2,Customer Support,1
3,UI & Design,1
4,Feature Requests,1


In [37]:
import nltk

print(nltk.__version__)

3.9.4


In [38]:
# download NLTK resources
import nltk

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\derese\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\derese\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\derese\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [39]:
import sys

print(sys.executable)

c:\Users\derese\fintech-review-analytics\venv\Scripts\python.exe


In [40]:
from src.nlp_pipeline import preprocess_text

In [41]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\derese\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [42]:
sample_df["processed_review"] = sample_df[
    "review"
].apply(preprocess_text)

In [43]:
sample_df[
    [
        "review",
        "processed_review"
    ]
].head(10)

,review,processed_review
0,wow,wow
1,Good application,good application
2,"Nice, but I can't get some recently transactio...",nice ca get recently transaction please try ke...
3,Very Secure but very poor interface and limite...,secure poor interface limited history service ...
4,very nice 100%,nice
5,good,good
6,Good to use,good use
7,cbe,cbe
8,Cbe,cbe
9,best and secured,best secured


In [44]:
print(sample_df.columns)

Index(['review_id', 'review', 'rating', 'date', 'bank', 'source',
       'sentiment_label', 'sentiment_score', 'processed_review'],
      dtype='object')


In [45]:
sample_df["theme"] = sample_df[
    "review"
].apply(identify_theme)

In [46]:
final_results = sample_df[
    [
        "review_id",
        "review",
        "sentiment_label",
        "sentiment_score",
        "theme"
    ]
].copy()

In [47]:
final_results.columns = [
    "review_id",
    "review_text",
    "sentiment_label",
    "sentiment_score",
    "identified_theme"
]

In [48]:
final_results.to_csv(
    "../data/raw/task2_final_results.csv",
    index=False
)

print("Task 2 final results saved successfully.")

Task 2 final results saved successfully.


In [72]:
import pandas as pd

reviews_final_for_db = pd.read_csv(
    "../data/raw/reviews_final_for_db.csv"
)

In [76]:
reviews_final_for_db[
    ["bank_id", "bank"]
].head()

,bank_id,bank
0,1,Commercial Bank of Ethiopia
1,1,Commercial Bank of Ethiopia
2,1,Commercial Bank of Ethiopia
3,1,Commercial Bank of Ethiopia
4,1,Commercial Bank of Ethiopia


In [82]:
reviews_final_for_db["sentiment_label"].unique()

array(['positive', 'negative'], dtype=object)